# 5. Product Analysis

## Objective

Analyze product performance to identify:

- Top-performing products
- Lowest-performing products
- Revenue by category
- Average selling price
- Product demand

In [1]:
import pandas as pd 
import sqlite3
# Connect to the SQLite database
conn = sqlite3.connect("supply_chain.db")

In [2]:
pd.read_sql("""
SELECT

SUM(f.delivery_qty * p.price_INR) AS total_revenue

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id;
""", conn)

,total_revenue
0,583964636


In [3]:
pd.read_sql("""
SELECT
    strftime('%m', order_placement_date) AS Month,
    SUM(delivery_qty * price_INR) AS Revenue
FROM fact_order_line f
JOIN dim_products p
ON f.product_id = p.product_id
GROUP BY Month
ORDER BY Month;
""", conn)

,Month,Revenue
0,None,583964636


In [4]:
## Top 10 Products by Revenue
pd.read_sql("""
SELECT

    p.product_id,
    p.product_name,
    p.category,

    SUM(f.delivery_qty) AS total_quantity_sold,

    ROUND(
        SUM(f.delivery_qty * p.price_INR),
        2
    ) AS revenue

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY

    p.product_id,
    p.product_name,
    p.category

ORDER BY revenue DESC

LIMIT 10;
""", conn)

,product_id,product_name,category,total_quantity_sold,revenue
0,25891203,AM Butter 500,Dairy,406359,121907700.0
1,25891501,AM Biscuits 750,Food,231243,69372900.0
2,25891101,AM Milk 500,Dairy,515147,64393375.0
3,25891202,AM Butter 250,Dairy,383200,57480000.0
4,25891502,AM Biscuits 500,Food,228596,45719200.0
5,25891601,AM Tea 500,beverages,165055,37137375.0
6,25891102,AM Milk 250,Dairy,532026,32985612.0
7,25891503,AM Biscuits 250,Food,235907,23590700.0
8,25891201,AM Butter 100,Dairy,380699,22841940.0
9,25891401,AM Curd 250,Dairy,444665,22233250.0


In [5]:
## Bottom 10 Products by Revenue
pd.read_sql("""
SELECT

    p.product_id,
    p.product_name,
    p.category,

    ROUND(
        SUM(f.delivery_qty * p.price_INR),
        2
    ) AS revenue

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY

    p.product_id,
    p.product_name,
    p.category

ORDER BY revenue ASC

LIMIT 10;
""", conn)

,product_id,product_name,category,revenue
0,25891403,AM Curd 50,Dairy,4576390.0
1,25891303,AM Ghee 100,Dairy,6810030.0
2,25891603,AM Tea 100,beverages,7101765.0
3,25891402,AM Curd 100,Dairy,8985100.0
4,25891302,AM Ghee 150,Dairy,10337625.0
5,25891103,AM Milk 100,Dairy,12547550.0
6,25891301,AM Ghee 250,Dairy,17732700.0
7,25891602,AM Tea 250,beverages,18211424.0
8,25891401,AM Curd 250,Dairy,22233250.0
9,25891201,AM Butter 100,Dairy,22841940.0


In [6]:
## Revenue by category
pd.read_sql("""
SELECT

    p.category,

    ROUND(
        SUM(f.delivery_qty * p.price_INR),
        2
    ) AS total_revenue

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY
    p.category

ORDER BY total_revenue DESC;
""", conn)


,category,total_revenue
0,Dairy,382831272.0
1,Food,138682800.0
2,beverages,62450564.0


In [7]:
## Total Quantity Sold by Product
pd.read_sql("""
SELECT

    p.product_id,
    p.product_name,

    SUM(f.delivery_qty) AS quantity_sold

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY

    p.product_id,
    p.product_name

ORDER BY quantity_sold DESC;
""", conn)

,product_id,product_name,quantity_sold
0,25891102,AM Milk 250,532026
1,25891101,AM Milk 500,515147
2,25891103,AM Milk 100,501902
3,25891403,AM Curd 50,457639
4,25891402,AM Curd 100,449255
5,25891401,AM Curd 250,444665
6,25891203,AM Butter 500,406359
7,25891202,AM Butter 250,383200
8,25891201,AM Butter 100,380699
9,25891503,AM Biscuits 250,235907


In [8]:
## Average Selling Price by Category
pd.read_sql("""
SELECT

    category,

    ROUND(
        AVG(price_INR),
        2
    ) AS average_price

FROM dim_products

GROUP BY category

ORDER BY average_price DESC;
""", conn)

,category,average_price
0,Food,200.00
1,beverages,127.33
2,Dairy,104.33


In [9]:
## Product Demand
pd.read_sql("""
SELECT

    p.product_id,
    p.product_name,

    COUNT(*) AS total_orders

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY

    p.product_id,
    p.product_name

ORDER BY total_orders DESC;
""", conn)

,product_id,product_name,total_orders
0,25891203,AM Butter 500,1409
1,25891503,AM Biscuits 250,1377
2,25891102,AM Milk 250,1374
3,25891502,AM Biscuits 500,1363
4,25891601,AM Tea 500,1361
5,25891301,AM Ghee 250,1360
6,25891403,AM Curd 50,1356
7,25891501,AM Biscuits 750,1355
8,25891202,AM Butter 250,1352
9,25891602,AM Tea 250,1349


In [10]:
## Product Performance Ranking
pd.read_sql("""
SELECT

    p.product_name,

    p.category,

    ROUND(
        SUM(f.delivery_qty * p.price_INR),
        2
    ) AS revenue,

    RANK() OVER(
        ORDER BY
            SUM(f.delivery_qty * p.price_INR) DESC
    ) AS revenue_rank

FROM fact_order_line f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY

    p.product_name,
    p.category;
""", conn)

,product_name,category,revenue,revenue_rank
0,AM Butter 500,Dairy,121907700.0,1
1,AM Biscuits 750,Food,69372900.0,2
2,AM Milk 500,Dairy,64393375.0,3
3,AM Butter 250,Dairy,57480000.0,4
4,AM Biscuits 500,Food,45719200.0,5
5,AM Tea 500,beverages,37137375.0,6
6,AM Milk 250,Dairy,32985612.0,7
7,AM Biscuits 250,Food,23590700.0,8
8,AM Butter 100,Dairy,22841940.0,9
9,AM Curd 250,Dairy,22233250.0,10


## Business Recommendations

1. Maintain high inventory levels for top-selling Dairy products to reduce stock-out risk.

2. Continue investing in premium Food products, as they deliver the highest revenue per unit.

3. Evaluate pricing and marketing strategies for low-performing products to improve their contribution.

4. Focus demand forecasting efforts on high-volume products such as Milk and Butter to optimize procurement and production planning.

5. Monitor category performance regularly to balance profitability (Food) with volume-driven growth (Dairy).